In [36]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import re
from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier

In [45]:
train_df = pd.read_csv('/content/drive/MyDrive/training_data_binary.csv') #load train dataset
test_df = pd.read_csv('/content/drive/MyDrive/testing_data_binary.csv') #load test dataset
X_train = train_df['prompt']
y_train = train_df['jailbreak']
X_val = test_df['prompt']
y_val = test_df['jailbreak']

In [46]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

1. Logistic Regression

In [15]:
model_LR = LogisticRegression(random_state=0, solver='saga', penalty='elasticnet', l1_ratio=0.2, max_iter=8000)

In [30]:
model_LR.fit(X_train_tfidf, y_train)

LogisticRegression(l1_ratio=0.2, max_iter=8000, penalty='elasticnet',
                   random_state=0, solver='saga')

In [31]:
y_pred = model_LR.predict(X_val_tfidf)
accuracy_LR = accuracy_score(y_val, y_pred)
f1_LR = f1_score(y_val, y_pred)
cm_LR = confusion_matrix(y_val, y_pred)

In [32]:
print(f'Accuracy: {accuracy_LR}')
print(f'F1 Score: {f1_LR}')
print(f'Confusion Matrix:\n{cm_LR}')

Accuracy: 0.8604992657856094
F1 Score: 0.8263254113345521
Confusion Matrix:
[[360  41]
 [ 54 226]]


2. Extra Trees Classifier

In [28]:
model_extraTrees = ExtraTreesClassifier(n_estimators=100, random_state=0)

In [33]:
model_extraTrees.fit(X_train_tfidf, y_train)

ExtraTreesClassifier(random_state=0)

In [34]:
y_pred_extraTrees = model_extraTrees.predict(X_val_tfidf)
accuracy_et = accuracy_score(y_val, y_pred_extraTrees)
f1_et = f1_score(y_val, y_pred_extraTrees)
cm_et = confusion_matrix(y_val, y_pred_extraTrees)

In [35]:
print(f'Accuracy: {accuracy_et}')
print(f'F1 Score: {f1_et}')
print(f'Confusion Matrix:\n{cm_et}')

Accuracy: 0.8663729809104258
F1 Score: 0.8305400372439479
Confusion Matrix:
[[367  34]
 [ 57 223]]


3. LGBM

In [38]:
model_LGBM = LGBMClassifier(n_jobs=-1)
model_LGBM.fit(X_train_tfidf, y_train)

[LightGBM] [Info] Number of positive: 2250, number of negative: 3198
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.515652 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 132146
[LightGBM] [Info] Number of data points in the train set: 5448, number of used features: 3492
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.412996 -> initscore=-0.351595
[LightGBM] [Info] Start training from score -0.351595


LGBMClassifier(n_jobs=-1)

In [39]:
y_pred_LGBM = model_LGBM.predict(X_val_tfidf)
accuracy_LGBM = accuracy_score(y_val, y_pred_LGBM)
f1_LGBM = f1_score(y_val, y_pred_LGBM)
cm_LGBM = confusion_matrix(y_val, y_pred_LGBM)

In [40]:
print(f'Accuracy: {accuracy_LGBM}')
print(f'F1 Score: {f1_LGBM}')
print(f'Confusion Matrix:\n{cm_LGBM}')

Accuracy: 0.8604992657856094
F1 Score: 0.8217636022514071
Confusion Matrix:
[[367  34]
 [ 61 219]]


Classification Reports

In [41]:
from sklearn.metrics import classification_report

In [44]:
print(f"Classification Report LR:\n{classification_report(y_val, y_pred)}")
print(f"Classification Report ET:\n{classification_report(y_val, y_pred_extraTrees)}")
print(f"Classification Report LGBM:\n{classification_report(y_val, y_pred_LGBM)}")

Classification Report LR:
              precision    recall  f1-score   support

           0       0.87      0.90      0.88       401
           1       0.85      0.81      0.83       280

    accuracy                           0.86       681
   macro avg       0.86      0.85      0.85       681
weighted avg       0.86      0.86      0.86       681

Classification Report ET:
              precision    recall  f1-score   support

           0       0.87      0.92      0.89       401
           1       0.87      0.80      0.83       280

    accuracy                           0.87       681
   macro avg       0.87      0.86      0.86       681
weighted avg       0.87      0.87      0.87       681

Classification Report LGBM:
              precision    recall  f1-score   support

           0       0.86      0.92      0.89       401
           1       0.87      0.78      0.82       280

    accuracy                           0.86       681
   macro avg       0.86      0.85      0.85     